# Поиск правила для частых покупателей

In [ ]:
'''
Что уже известно:
- у клиентов с 2–3 днями заказов target почти предопределён: если день «последний заказ + средний интервал»
  попадает в зону 25..92 дня после отсечения, доля покупок 0.86–0.90, вне зоны 0.23–0.30;
- для 4+ дней заказов то же правило по среднему, медианному и последнему интервалу не работает;
- у частых покупателей странность: купившие недавно покупают реже, чем «опаздывающие».

Гипотеза: правило действует не на отдельные заказы, а на визиты — серии заказов, идущих подряд с малым разрывом.
У частого покупателя визитов мало, интервал между ними длинный, и только что закончивший серию клиент
действительно далёк от следующего визита.

Что делаем здесь (обучение только в последнем разделе, и оно маленькое):
- склеиваем дни с заказами в визиты при разных порогах и проверяем правило по визитам;
- перебираем другие оценки периода: квантили интервалов, последние и первые k интервалов, только длинные интервалы;
- смотрим на истории конкретных клиентов;
- проверяем, видит ли модель на «сырых» датах заказов что-то, чего не видят формулы.
'''
import os
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))

SEED = 42
GAP_DAYS = 14
ZONE_LO, ZONE_HI = 25, 92
N_EDGES = [0, 1, 2, 3, 5, 9, 20, 10_000]
N_LABELS = ["1", "2", "3", "4-5", "6-9", "10-20", "21+"]


def fast_ap(target, score):
    order = np.argsort(-np.asarray(score), kind="mergesort")
    t = np.asarray(target)[order]
    k = int(np.ceil(len(t) * 0.20))
    top = t[:k]
    return float((np.cumsum(top) / np.arange(1, k + 1) * top).sum() / max(min(k, int(t.sum())), 1))

## Данные

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
orders = pd.read_parquet(DATA_DIR / "orders.parquet", columns=["customer_id", "order_id", "order_date"])
orders["order_date"] = pd.to_datetime(orders["order_date"])

cs = train.copy()
cs["slice"] = cs["scoring_date"].dt.strftime("%Y-%m-%d")
cs["cutoff"] = cs["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
cs = cs.set_index("customer_id")

# одна строка — один день с заказами клиента из train; d — дней до отсечения, самые свежие дни идут первыми
days = orders[orders["customer_id"].isin(cs.index)].drop_duplicates(["customer_id", "order_date"]).copy()
days["d"] = (days["customer_id"].map(cs["cutoff"]) - days["order_date"]).dt.days
days = days[["customer_id", "d"]].sort_values(["customer_id", "d"]).reset_index(drop=True)
days["gap"] = days.groupby("customer_id")["d"].diff()          # интервал до предыдущего по времени дня с заказами
days["rk"] = days.groupby("customer_id").cumcount()            # 0 — самый свежий день

cs["n_days"] = days.groupby("customer_id").size()
cs["recency"] = days.groupby("customer_id")["d"].min()
cs["tenure"] = days.groupby("customer_id")["d"].max()
cs["n_bin"] = pd.cut(cs["n_days"], N_EDGES, labels=N_LABELS)
print("клиентов:", len(cs), "| дней с заказами:", len(days))
display(cs.groupby("n_bin", observed=True)["target"].agg(["size", "mean"]).T)


def zone_tables(t_by_name, group="n_bin"):
    # t_by_name: {название оценки: ожидаемый день следующего заказа, индекс — customer_id}
    inside_rate, outside_rate, share = {}, {}, {}
    for name, t in t_by_name.items():
        t = t.reindex(cs.index)
        ok = t.notna()
        inside = t.gt(ZONE_LO) & t.le(ZONE_HI)
        inside_rate[name] = cs[inside].groupby(group, observed=True)["target"].mean()
        outside_rate[name] = cs[ok & ~inside].groupby(group, observed=True)["target"].mean()
        share[name] = inside[ok].groupby(cs.loc[ok, group], observed=True).mean()
    print(f"Доля target = 1, когда ожидаемый день в зоне {ZONE_LO}..{ZONE_HI}:")
    display(pd.DataFrame(inside_rate).T)
    print("Доля target = 1 вне зоны:")
    display(pd.DataFrame(outside_rate).T)
    print("Доля клиентов в зоне:")
    display(pd.DataFrame(share).T)

## Визиты

In [ ]:
# Визит — серия дней с заказами, где соседние дни отстоят не больше чем на delta дней.
# Период считаем двумя способами: по последним дням визитов («концы») и по первым («начала»).
DELTAS = [0, 1, 3, 7, 14, 21, 30, 45]
visit_t, visit_n = {}, {}
for delta in DELTAS:
    new = days["gap"].isna() | (days["gap"] > delta)
    v = days.assign(visit=new.groupby(days["customer_id"]).cumsum())
    vs = v.groupby(["customer_id", "visit"])["d"].agg(end="min", start="max").reset_index()   # end — самый свежий день визита
    g = vs.groupby("customer_id")
    n_vis = g.size()
    last, first = vs.drop_duplicates("customer_id").set_index("customer_id"), vs.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    period_end = (first["end"] - last["end"]) / (n_vis - 1).replace(0, np.nan)
    period_start = (first["start"] - last["start"]) / (n_vis - 1).replace(0, np.nan)
    visit_t[f"delta={delta}, концы"] = period_end - last["end"]        # от последнего заказа
    visit_t[f"delta={delta}, начала"] = period_start - last["start"]   # от начала последнего визита
    visit_n[delta] = n_vis
    cs[f"n_vis_{delta}"] = n_vis

print("Среднее число визитов по группам числа дней с заказами:")
display(cs.groupby("n_bin", observed=True)[[f"n_vis_{d}" for d in DELTAS]].mean())
print("=== По числу дней с заказами")
zone_tables(visit_t)

for delta in [7, 14, 30]:
    cs["vis_bin"] = pd.cut(cs[f"n_vis_{delta}"], [0, 1, 2, 3, 5, 9, 10_000], labels=["1", "2", "3", "4-5", "6-9", "10+"])
    print(f"=== delta = {delta}: по числу визитов")
    zone_tables({k: v for k, v in visit_t.items() if k.startswith(f"delta={delta},")}, group="vis_bin")

# Профиль с шагом 10 дней для клиентов с 6+ днями заказов: есть ли вообще зона
freq = cs[cs["n_days"] >= 6]
bins = [-800, -60, -30, 0, 15, 25, 40, 55, 70, 85, 92, 105, 130, 180, 800]
prof = {name: freq.groupby(pd.cut(t.reindex(freq.index), bins), observed=False)["target"].mean() for name, t in visit_t.items() if "концы" in name}
print("Клиенты с 6+ днями заказов: доля target = 1 по ожидаемому дню следующего визита")
display(pd.DataFrame(prof))
print("Число клиентов:")
display(pd.DataFrame({name: freq.groupby(pd.cut(t.reindex(freq.index), bins), observed=False)["target"].size() for name, t in visit_t.items() if "концы" in name}))

## Другие оценки периода

In [ ]:
g = days["gap"]
c = days["customer_id"]
rk_old = days.groupby("customer_id")["rk"].transform("max") - days["rk"]   # 0 — самый старый день
r = cs["recency"]
est = {"среднее": g.groupby(c).mean()}
for q in [0.25, 0.5, 0.75, 0.9, 1.0]:
    est[f"квантиль {q}"] = g.groupby(c).quantile(q)
for k in [1, 2, 3, 5]:
    est[f"последние {k}"] = g.where(days["rk"].between(1, k)).groupby(c).mean()
for k in [1, 2, 3]:
    est[f"первые {k}"] = g.where(rk_old < k).groupby(c).mean()
for thr in [3, 7, 14, 30, 60]:
    est[f"только интервалы > {thr}"] = g.where(g > thr).groupby(c).mean()
est["стаж / число дней"] = cs["tenure"] / cs["n_days"]
est["(стаж - давность) / 2"] = (cs["tenure"] - cs["recency"]) / 2
est["(стаж - давность) / 3"] = (cs["tenure"] - cs["recency"]) / 3
zone_tables({name: p.reindex(cs.index) - r for name, p in est.items()})

# То же в долях периода: какой доле периода равна давность, без привязки к зоне
print("Доля target = 1 по отношению давности к периоду «только интервалы > 14», по числу дней с заказами:")
u = (r / est["только интервалы > 14"].reindex(cs.index)).rename("u")
display(cs.assign(u_bin=pd.cut(u, [-0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5, 2, 3, 1e9]))
        .pivot_table(index="u_bin", columns="n_bin", values="target", aggfunc="mean", observed=True))

## Истории конкретных клиентов

In [ ]:
# Числа — дни до отсечения, от самого свежего заказа к самому старому. Целевое окно начинается через 14 дней после отсечения.
hist = days.groupby("customer_id")["d"].apply(list)
rng = np.random.default_rng(SEED)
for lo, hi in [(4, 5), (6, 9), (10, 20)]:
    for target in [1, 0]:
        pool = cs[(cs["n_days"].between(lo, hi)) & (cs["target"] == target)].index.to_numpy()
        print(f"\n--- дней с заказами {lo}–{hi}, target = {target}")
        for cid in rng.choice(pool, size=min(12, len(pool)), replace=False):
            print(" ".join(f"{d:>3d}" for d in hist[cid][:22]))

## Модель на сырых датах

In [ ]:
# Если модель, которой даны сами даты заказов, предсказывает группу 4+ заметно лучше модели на формулах,
# значит, во времени заказов есть правило, которое формулы не выражают.
K_LAST, K_FIRST = 15, 3
wide_last = days[days["rk"] < K_LAST].pivot(index="customer_id", columns="rk", values="d")
wide_last.columns = [f"d_last_{i + 1}" for i in wide_last.columns]
old = days.assign(rk_old=rk_old)
wide_first = old[old["rk_old"] < K_FIRST].pivot(index="customer_id", columns="rk_old", values="d")
wide_first.columns = [f"d_first_{i + 1}" for i in wide_first.columns]
gap_last = days[days["rk"].between(1, 10)].pivot(index="customer_id", columns="rk", values="gap")
gap_last.columns = [f"gap_last_{i}" for i in gap_last.columns]

F = cs[["n_days", "recency", "tenure"]].copy()
F["gap_mean"] = est["среднее"]
F["gap_median"] = est["квантиль 0.5"]
F["gap_max"] = est["квантиль 1.0"]
F["gap_long14"] = est["только интервалы > 14"]
F["n_vis_14"] = cs["n_vis_14"]
for name in ["gap_mean", "gap_median", "gap_long14"]:
    F[f"t_{name}"] = F[name] - F["recency"]
    F[f"u_{name}"] = F["recency"] / F[name]
F["formula"] = F["n_days"] / (F["recency"] + 30)
F["t_visits_14"] = visit_t["delta=14, концы"]
SETS = {
    "формулы": list(F.columns),
    "сырые даты": ["n_days"] + list(wide_last.columns) + list(wide_first.columns) + list(gap_last.columns),
}
X = F.join(wide_last).join(wide_first).join(gap_last)
SETS["формулы + сырые даты"] = list(X.columns)
y = cs["target"].to_numpy()
sl = cs["slice"].to_numpy()

params = dict(objective="binary", learning_rate=0.05, n_estimators=3000, num_leaves=31, min_child_samples=100, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
oof = {}
for name, feats in SETS.items():
    p = np.zeros(len(X))
    for a, b in skf.split(X, cs["slice"] + "_" + cs["target"].astype(str)):
        m = lgb.LGBMClassifier(**params).fit(X.iloc[a][feats], y[a], eval_set=[(X.iloc[b][feats], y[b])], callbacks=[lgb.early_stopping(100, verbose=False)])
        p[b] = m.predict_proba(X.iloc[b][feats])[:, 1]
    oof[name] = p

rows = {}
for name, p in oof.items():
    row = {"AP@20% по срезам": float(np.mean([fast_ap(y[sl == s], p[sl == s]) for s in np.unique(sl)]))}
    for nb in N_LABELS:
        m = (cs["n_bin"] == nb).to_numpy()
        if m.sum() > 50 and 0 < y[m].mean() < 1:
            row[f"AUC, {nb} дн."] = roc_auc_score(y[m], p[m])
    rows[name] = row
print("Качество трёх наборов признаков: общее и внутри групп по числу дней с заказами")
display(pd.DataFrame(rows).T)